# CertifyTube - Engagement Source Mirror (Viva Notebook)

This notebook copies the engagement module source into one place for presentation.

How to present:
1. Explain the file role (short note above each code block).
2. Walk through the key functions only (you do not need to execute every cell).
3. Connect each part to API behavior (`/engagement/analyze/xgboost`, `/engagement/analyze/ebm`).

Note:
- This notebook is a presentation mirror of source files.
- Original project files remain unchanged in their folders.

## Quick Map
- `common/` - shared validation, split, explanation logic
- `contracts/` - feature contract enforcement
- `utils/` - helper functions
- `xgboost/` - training, evaluation, inference, SHAP explanation
- `ebm/` - training, evaluation, inference, native explanation

## `verification/engagement/common/behavior_map.py`
**What it does:** Maps each feature to a human-readable behavior category used in explanations.

In [ ]:
# SOURCE MIRROR FROM: verification/engagement/common/behavior_map.py
# PURPOSE: Maps each feature to a human-readable behavior category used in explanations.
# VIVA TIP: focus on function signatures, inputs/outputs, and why this file exists.

from __future__ import annotations

from typing import Dict

FEATURE_TO_BEHAVIOR: Dict[str, str] = {
    # Coverage / completion
    "session_duration_sec": "coverage",
    "video_duration_sec": "coverage",
    "watch_time_ratio": "coverage",
    "completion_ratio": "coverage",
    "watch_time_sec": "coverage",
    "completed_flag": "coverage",
    "last_position_sec": "coverage",

    # Skipping (forward seeks + skip ratios)
    "num_seek_forward": "skipping",
    "num_seek": "skipping",
    "total_seek_forward_sec": "skipping",
    "avg_seek_forward_sec": "skipping",
    "largest_forward_seek_sec": "skipping",
    "seek_forward_ratio": "skipping",
    "skip_time_ratio": "skipping",
    "early_skip_flag": "skipping",
    "skim_flag": "skipping",

    # Rewatching (backward seeks + rewatch ratios)
    "num_seek_backward": "rewatching",
    "total_seek_backward_sec": "rewatching",
    "avg_seek_backward_sec": "rewatching",
    "largest_backward_seek_sec": "rewatching",
    "seek_backward_ratio": "rewatching",
    "rewatch_time_ratio": "rewatching",
    "rewatch_to_skip_ratio": "rewatching",
    "deep_flag": "rewatching",

    # Reflective pausing
    "num_pause": "reflective_pausing",
    "total_pause_duration_sec": "reflective_pausing",
    "avg_pause_duration_sec": "reflective_pausing",
    "median_pause_duration_sec": "reflective_pausing",
    "long_pause_count": "reflective_pausing",
    "long_pause_ratio": "reflective_pausing",
    "pause_freq_per_min": "reflective_pausing",

    # Speed watching / playback behaviour
    "avg_playback_rate_when_playing": "speed_watching",
    "fast_ratio": "speed_watching",
    "slow_ratio": "speed_watching",
    "playback_speed_variance": "speed_watching",
    "num_ratechange": "speed_watching",
    "time_at_speed_lt1x_sec": "speed_watching",
    "time_at_speed_1x_sec": "speed_watching",
    "time_at_speed_gt1x_sec": "speed_watching",
    "unique_speed_levels": "speed_watching",

    # Attention / quality signals
    "attention_index": "attention_consistency",
    "engagement_velocity": "attention_consistency",
    "seek_density_per_min": "attention_consistency",
    "seek_jump_std_sec": "attention_consistency",
    "first_seek_time_sec": "attention_consistency",
    "play_pause_ratio": "attention_consistency",

    # Buffering (not learner intent, but affects experience)
    "num_buffering_events": "playback_quality",
    "buffering_time_sec": "playback_quality",
    "buffering_freq_per_min": "playback_quality",
}


def get_behavior(feature_name: str) -> str:
    return FEATURE_TO_BEHAVIOR.get(feature_name, "other")

## `verification/engagement/common/split.py`
**What it does:** Implements grouped train/test split to reduce leakage across users.

In [ ]:
# SOURCE MIRROR FROM: verification/engagement/common/split.py
# PURPOSE: Implements grouped train/test split to reduce leakage across users.
# VIVA TIP: focus on function signatures, inputs/outputs, and why this file exists.

from __future__ import annotations

from typing import Optional, Tuple
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold


def split_train_test(
    df: pd.DataFrame,
    label_col: str,
    group_col: Optional[str] = None,
    test_size: float = 0.2,
    random_state: int = 42,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Stratified + Grouped split:
      - preserves label distribution
      - prevents leakage between groups (e.g., user_id)

    Uses StratifiedGroupKFold with n_splits ≈ 1/test_size.
    For 80/20 => n_splits=5 (exact).
    """
    if label_col not in df.columns:
        raise ValueError(f"Label column '{label_col}' not found.")
    if not group_col or group_col not in df.columns:
        raise ValueError("group_col must be provided and exist in df for StratifiedGroupKFold.")

    n_splits = int(round(1.0 / test_size))
    if abs((1.0 / n_splits) - test_size) > 1e-6:
        raise ValueError(f"test_size={test_size} not supported cleanly; use values like 0.2, 0.25, 0.1")

    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    X_dummy = df.drop(columns=[label_col])
    y = df[label_col].astype(int)
    groups = df[group_col].astype(str)

    # Use first fold as test for determinism
    train_idx, test_idx = next(sgkf.split(X_dummy, y, groups))
    train_df = df.iloc[train_idx].reset_index(drop=True)
    test_df = df.iloc[test_idx].reset_index(drop=True)
    return train_df, test_df

## `verification/engagement/common/text_explainer.py`
**What it does:** Builds learner-friendly explanation text from top contributors.

In [ ]:
# SOURCE MIRROR FROM: verification/engagement/common/text_explainer.py
# PURPOSE: Builds learner-friendly explanation text from top contributors.
# VIVA TIP: focus on function signatures, inputs/outputs, and why this file exists.

from __future__ import annotations

from typing import Dict, List, Optional

from app.core.settings import settings
from verification.engagement.common.behavior_map import get_behavior

# ---------------------------------------------------------------------------
# Friendly, human-readable labels for each behavior category
# ---------------------------------------------------------------------------

BEHAVIOR_LABELS = {
    "coverage": "content coverage",
    "attention_consistency": "attention consistency",
    "rewatching": "active review",
    "reflective_pausing": "reflective pauses",
    "speed_watching": "playback pacing",
    "skipping": "continuous viewing",
    "playback_quality": "stable playback",
    "other": "learning behavior",
}

# ---------------------------------------------------------------------------
# Per-feature reasons (used for the contributor-level `reason` field)
# ---------------------------------------------------------------------------

POSITIVE_REASON = {
    "coverage": "You covered most of the lesson.",
    "attention_consistency": "Your attention looked consistent.",
    "rewatching": "You revisited important parts.",
    "reflective_pausing": "Your pauses looked reflective.",
    "speed_watching": "Your playback pace supported understanding.",
    "skipping": "You avoided excessive skipping.",
    "playback_quality": "Playback stayed stable enough for learning.",
    "other": "Your learning behavior supported understanding.",
}

NEGATIVE_REASON = {
    "coverage": "Coverage was not strong enough yet.",
    "attention_consistency": "Attention looked inconsistent in parts.",
    "rewatching": "There was limited review of key sections.",
    "reflective_pausing": "Pausing behavior did not show enough reflection.",
    "speed_watching": "Playback pacing may have reduced comprehension.",
    "skipping": "Frequent skipping reduced continuity.",
    "playback_quality": "Playback interruptions affected continuity.",
    "other": "Some behavior signals reduced engagement quality.",
}

# ---------------------------------------------------------------------------
# Conversational messages for the top-level `explanation` field.
# These are friendly, empathetic messages — like a tutor talking to a student.
# ---------------------------------------------------------------------------

CONGRATS_MESSAGES = {
    "coverage": "You watched most of the lesson thoroughly — great commitment!",
    "attention_consistency": "Your attention stayed consistent throughout — well done!",
    "rewatching": "You went back to review key parts — that shows real effort to understand!",
    "reflective_pausing": "Your thoughtful pausing shows you were really processing the material!",
    "speed_watching": "You kept a comfortable pace that supports real understanding!",
    "skipping": "You followed the lesson flow without jumping around — solid focus!",
    "playback_quality": "Your playback was smooth and uninterrupted — perfect conditions for learning!",
    "other": "Your overall learning behavior shows strong engagement!",
}

SORRY_MESSAGES = {
    "skipping": (
        "It looks like you skipped through some sections. "
        "Try watching the full lesson flow — the concepts build on each other!"
    ),
    "coverage": (
        "It seems like some parts of the lesson were missed. "
        "Watching the complete video helps build a stronger understanding."
    ),
    "speed_watching": (
        "Playing at high speed can make it harder to absorb the material. "
        "Try a comfortable pace where you can follow along."
    ),
    "attention_consistency": (
        "Your attention seemed to drift in parts. "
        "Try a focused session without distractions — it makes a big difference!"
    ),
    "reflective_pausing": (
        "Consider pausing after key ideas to let them sink in. "
        "Brief reflection helps lock in understanding."
    ),
    "rewatching": (
        "Rewatching tricky sections can really help. "
        "Next time, try going back to the parts that felt unclear."
    ),
    "playback_quality": (
        "Buffering interruptions made it harder to focus. "
        "A more stable connection would help you stay in the flow."
    ),
    "other": (
        "Some of your viewing patterns suggest you may not have fully engaged with the lesson. "
        "Try watching at a steady pace and take brief pauses to reflect."
    ),
}


# ---------------------------------------------------------------------------
# Internal helpers
# ---------------------------------------------------------------------------

def _top_behaviors(rows: List[Dict[str, float]], limit: int = 2) -> List[str]:
    ordered: List[str] = []
    for row in rows:
        feature = row.get("feature")
        if not feature:
            continue
        behavior = get_behavior(str(feature))
        if behavior not in ordered:
            ordered.append(behavior)
        if len(ordered) >= limit:
            break
    return ordered


def _format_score(engagement_score: Optional[float]) -> str:
    if engagement_score is None:
        return "N/A"
    bounded = max(0.0, min(1.0, float(engagement_score)))
    return f"{bounded * 100:.0f}%"


# ---------------------------------------------------------------------------
# Public API
# ---------------------------------------------------------------------------

def build_feature_reason(feature: str, feature_value: float, shap_value: float) -> str:
    """Return a short reason for a single contributor (used in the contributor list)."""
    _ = feature_value  # intentionally unused to avoid exposing gameable thresholds.
    behavior = get_behavior(feature)
    if shap_value >= 0:
        return POSITIVE_REASON.get(behavior, POSITIVE_REASON["other"])
    return NEGATIVE_REASON.get(behavior, NEGATIVE_REASON["other"])


def build_user_explanation(
    shap_top_negative: List[Dict[str, float]],
    shap_top_positive: List[Dict[str, float]],
    engagement_score: Optional[float] = None,
) -> str:
    """Build a friendly, conversational explanation for the learner.

    - **Engaged**: Congratulatory message highlighting what they did well.
    - **Not engaged**: Empathetic message explaining the common reason and an actionable tip.
    """
    threshold = settings.engagement_threshold
    is_engaged = engagement_score is not None and engagement_score >= threshold
    score_text = _format_score(engagement_score)

    positive_behaviors = _top_behaviors(shap_top_positive, limit=2)
    negative_behaviors = _top_behaviors(shap_top_negative, limit=2)

    if is_engaged:
        # Pick the strongest positive behavior for the congrats message
        top_behavior = positive_behaviors[0] if positive_behaviors else "other"
        congrats = CONGRATS_MESSAGES.get(top_behavior, CONGRATS_MESSAGES["other"])
        return (
            f"Congratulations! Your engagement score is {score_text}. "
            f"{congrats} Keep this up in your next session!"
        )

    # Not engaged — use the top negative behavior for the sorry message
    top_negative = negative_behaviors[0] if negative_behaviors else "other"
    sorry = SORRY_MESSAGES.get(top_negative, SORRY_MESSAGES["other"])
    return (
        f"Your engagement score is {score_text}. "
        f"{sorry}"
    )

## `verification/engagement/common/validate.py`
**What it does:** Validates request feature dictionaries and enforces numeric types.

In [ ]:
# SOURCE MIRROR FROM: verification/engagement/common/validate.py
# PURPOSE: Validates request feature dictionaries and enforces numeric types.
# VIVA TIP: focus on function signatures, inputs/outputs, and why this file exists.

from typing import Dict, List


class FeatureValidationError(Exception):
    pass


def validate_features(
    features: Dict[str, float],
    expected_columns: List[str],
):
    if not isinstance(features, dict):
        raise FeatureValidationError("Features must be a dictionary")

    missing = [c for c in expected_columns if c not in features]
    extra = [k for k in features.keys() if k not in expected_columns]

    if missing:
        raise FeatureValidationError(f"Missing features: {missing}")

    if extra:
        raise FeatureValidationError(f"Unexpected features: {extra}")

    # Type safety (basic)
    for k, v in features.items():
        if not isinstance(v, (int, float)):
            raise FeatureValidationError(f"Feature '{k}' must be numeric")

## `verification/engagement/contracts/contract.py`
**What it does:** Loads feature contract versions and validates payload keys against the contract.

In [ ]:
# SOURCE MIRROR FROM: verification/engagement/contracts/contract.py
# PURPOSE: Loads feature contract versions and validates payload keys against the contract.
# VIVA TIP: focus on function signatures, inputs/outputs, and why this file exists.

import json
from pathlib import Path
from typing import Dict, List

CONTRACTS_DIR = Path("verification/engagement/contracts")


class ContractError(Exception):
    pass


def load_contract(version: str) -> List[str]:
    if version != "v1.0":
        raise ContractError(f"Unsupported feature_version '{version}'. Expected 'v1.0'.")

    path = CONTRACTS_DIR / "feature_contract_v1.json"
    if not path.exists():
        raise ContractError("Feature contract file missing: feature_contract_v1.json")

    with open(path, "r") as f:
        contract = json.load(f)

    if contract.get("feature_version") != "v1.0":
        raise ContractError("Contract version mismatch inside contract file.")

    features = contract.get("features")
    if not isinstance(features, list) or not features:
        raise ContractError("Contract file must contain a non-empty 'features' list.")

    return features


def validate_payload(features: Dict[str, float], expected: List[str]) -> None:
    missing = [c for c in expected if c not in features]
    extra = [k for k in features.keys() if k not in expected]

    if missing:
        raise ContractError(f"Missing features (contract mismatch): {missing}")

    if extra:
        raise ContractError(f"Unexpected features (contract mismatch): {extra}")

    for k, v in features.items():
        if not isinstance(v, (int, float)):
            raise ContractError(f"Feature '{k}' must be numeric")

## `verification/engagement/contracts/feature_contract_v1.json`
**What it does:** Canonical feature contract listing all required engagement features.

```json
{
  "feature_version": "v1.0",
  "features": [
  "session_duration_sec",
  "video_duration_sec",
  "last_position_sec",
  "completed_flag",
  "watch_time_sec",
  "watch_time_ratio",
  "completion_ratio",
  "engagement_velocity",
  "num_pause",
  "total_pause_duration_sec",
  "avg_pause_duration_sec",
  "median_pause_duration_sec",
  "pause_freq_per_min",
  "long_pause_count",
  "long_pause_ratio",
  "num_seek",
  "num_seek_forward",
  "num_seek_backward",
  "total_seek_forward_sec",
  "total_seek_backward_sec",
  "avg_seek_forward_sec",
  "avg_seek_backward_sec",
  "largest_forward_seek_sec",
  "largest_backward_seek_sec",
  "seek_jump_std_sec",
  "seek_forward_ratio",
  "seek_backward_ratio",
  "skip_time_ratio",
  "rewatch_time_ratio",
  "rewatch_to_skip_ratio",
  "seek_density_per_min",
  "first_seek_time_sec",
  "early_skip_flag",
  "num_ratechange",
  "time_at_speed_lt1x_sec",
  "time_at_speed_1x_sec",
  "time_at_speed_gt1x_sec",
  "fast_ratio",
  "slow_ratio",
  "playback_speed_variance",
  "avg_playback_rate_when_playing",
  "unique_speed_levels",
  "num_buffering_events",
  "buffering_time_sec",
  "buffering_freq_per_min",
  "play_pause_ratio",
  "attention_index",
  "skim_flag",
  "deep_flag"
  ]
}

```

## `verification/engagement/utils/io.py`
**What it does:** Utility I/O helpers shared by training/inference scripts.

In [ ]:
# SOURCE MIRROR FROM: verification/engagement/utils/io.py
# PURPOSE: Utility I/O helpers shared by training/inference scripts.
# VIVA TIP: focus on function signatures, inputs/outputs, and why this file exists.

from __future__ import annotations
import json
from pathlib import Path
from typing import Any
import joblib


def load_json(path: str | Path) -> Any:
    path = Path(path)
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def save_json(path: str | Path, data: Any, indent: int = 2) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, indent=indent)


def load_joblib(path: str | Path) -> Any:
    return joblib.load(Path(path))


def save_joblib(path: str | Path, obj: Any) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(obj, path)

## `verification/engagement/utils/math.py`
**What it does:** Utility math helpers used by engagement pipelines.

In [ ]:
# SOURCE MIRROR FROM: verification/engagement/utils/math.py
# PURPOSE: Utility math helpers used by engagement pipelines.
# VIVA TIP: focus on function signatures, inputs/outputs, and why this file exists.

from __future__ import annotations


def clamp(x: float, lo: float, hi: float) -> float:
    return max(lo, min(hi, x))


def safe_div(a: float, b: float, default: float = 0.0) -> float:
    if b == 0:
        return default
    return a / b

## `verification/engagement/xgboost/training/train.py`
**What it does:** End-to-end XGBoost training pipeline with tuning, evaluation, and artifact export.

In [ ]:
# SOURCE MIRROR FROM: verification/engagement/xgboost/training/train.py
# PURPOSE: End-to-end XGBoost training pipeline with tuning, evaluation, and artifact export.
# VIVA TIP: focus on function signatures, inputs/outputs, and why this file exists.

from __future__ import annotations

import json
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import joblib
import xgboost as xgb
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

from verification.engagement.common.split import split_train_test

# =========================
# CONFIG
# =========================
DATA_PATH = Path("data/processed/sessions_features.csv")

LABEL_COL = "engagement_label"
GROUP_COL = "user_id"

ARTIFACTS_DIR = Path("verification/engagement/xgboost/artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

REPORTS_DIR = Path("reports")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.2  # 80/20

ENGAGEMENT_THRESHOLD = 0.85

DROP_COLS = [
    LABEL_COL,
    GROUP_COL,
    "session_id",
    "video_id",
    "video_title",
]


def _prepare_features(df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray, List[str], pd.DataFrame]:
    if LABEL_COL not in df.columns:
        raise ValueError(f"Missing label column: {LABEL_COL}")

    y = df[LABEL_COL].astype(int).values

    X = df.drop(columns=[c for c in DROP_COLS if c in df.columns], errors="ignore")
    X = X.select_dtypes(include=[np.number]).copy()

    # Median impute (0 can be a real signal)
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.fillna(X.median(numeric_only=True))

    feature_columns = list(X.columns)
    if not feature_columns:
        raise ValueError("No numeric features found after preprocessing.")

    return X.values, y, feature_columns, X


def _sample_params(rng: np.random.Generator) -> Dict:
    return {
        "objective": "binary:logistic",
        "eval_metric": ["auc", "logloss"],   # IMPORTANT: logloss helps diagnose overfit too
        "tree_method": "hist",
        "seed": RANDOM_STATE,

        "max_depth": int(rng.integers(3, 8)),
        "min_child_weight": float(rng.uniform(1, 10)),
        "eta": float(rng.uniform(0.01, 0.2)),
        "subsample": float(rng.uniform(0.6, 1.0)),
        "colsample_bytree": float(rng.uniform(0.6, 1.0)),
        "gamma": float(rng.uniform(0.0, 5.0)),
        "lambda": float(rng.uniform(0.1, 10.0)),
        "alpha": float(rng.uniform(0.0, 5.0)),
    }


def _make_cv_folds(df_train: pd.DataFrame, feature_columns: List[str]) -> List[Tuple[np.ndarray, np.ndarray]]:
    from sklearn.model_selection import StratifiedGroupKFold

    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    y = df_train[LABEL_COL].astype(int).values
    groups = df_train[GROUP_COL].astype(str).values
    X_dummy = df_train[feature_columns].values

    folds = []
    for tr_idx, va_idx in sgkf.split(X_dummy, y, groups):
        folds.append((tr_idx, va_idx))
    return folds


def _plot_learning_curves(evals_result: Dict, out_path: Path) -> None:
    """
    Overfitting/Underfitting evidence:
      - Train vs Test AUC
      - Train vs Test logloss
    """
    # evals_result structure:
    # evals_result["train"]["auc"] etc.
    train_auc = evals_result.get("train", {}).get("auc", [])
    test_auc = evals_result.get("test", {}).get("auc", [])
    train_ll = evals_result.get("train", {}).get("logloss", [])
    test_ll = evals_result.get("test", {}).get("logloss", [])

    if train_auc and test_auc:
        plt.figure()
        plt.plot(train_auc, label="Train AUC")
        plt.plot(test_auc, label="Test AUC")
        plt.xlabel("Boosting Rounds")
        plt.ylabel("AUC")
        plt.title("Learning Curve: Train vs Test AUC")
        plt.legend()
        plt.savefig(out_path / "learning_curve_auc.png", bbox_inches="tight")
        plt.close()

    if train_ll and test_ll:
        plt.figure()
        plt.plot(train_ll, label="Train Logloss")
        plt.plot(test_ll, label="Test Logloss")
        plt.xlabel("Boosting Rounds")
        plt.ylabel("Logloss")
        plt.title("Learning Curve: Train vs Test Logloss")
        plt.legend()
        plt.savefig(out_path / "learning_curve_logloss.png", bbox_inches="tight")
        plt.close()


def main():
    print("Loading dataset...")
    df = pd.read_csv(DATA_PATH)

    if LABEL_COL not in df.columns:
        raise ValueError(f"Label column '{LABEL_COL}' not found in dataset")
    if GROUP_COL not in df.columns:
        raise ValueError(f"Group column '{GROUP_COL}' not found in dataset")

    X_all, y_all, feature_columns, X_df = _prepare_features(df)

    clean_df = X_df.copy()
    clean_df[LABEL_COL] = y_all
    clean_df[GROUP_COL] = df[GROUP_COL].astype(str)
    clean_df["session_id"] = df["session_id"].astype(str) if "session_id" in df.columns else df.index.astype(str)

    print("Splitting train/test (80/20) with stratified grouped split...")
    train_df, test_df = split_train_test(
        clean_df,
        label_col=LABEL_COL,
        group_col=GROUP_COL,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
    )

    X_train = train_df[feature_columns].values
    y_train = train_df[LABEL_COL].values
    X_test = test_df[feature_columns].values
    y_test = test_df[LABEL_COL].values

    dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=feature_columns)
    dtest = xgb.DMatrix(X_test, label=y_test, feature_names=feature_columns)

    print("Tuning hyperparameters (random search + xgb.cv)...")
    rng = np.random.default_rng(RANDOM_STATE)
    folds = _make_cv_folds(train_df, feature_columns)

    n_trials = 40
    best = {"auc": -1.0, "params": None, "best_round": None}

    for t in range(n_trials):
        params = _sample_params(rng)

        cv = xgb.cv(
            params=params,
            dtrain=dtrain,
            num_boost_round=5000,
            folds=folds,
            early_stopping_rounds=50,
            verbose_eval=False,
            as_pandas=True,
        )

        best_round = len(cv)
        auc_mean = float(cv["test-auc-mean"].iloc[-1])

        if auc_mean > best["auc"]:
            best = {"auc": auc_mean, "params": params, "best_round": best_round}
            print(f"  Trial {t+1}/{n_trials}: NEW BEST AUC={auc_mean:.4f}, rounds={best_round}")

    if best["params"] is None:
        raise RuntimeError("Tuning failed: no params selected.")

    print("Best CV AUC:", best["auc"])
    print("Best params:", best["params"])
    print("Best num_boost_round:", best["best_round"])

    # ---- Train final model and CAPTURE eval history for learning curves ----
    print("Training final booster on full train split...")
    evals_result: Dict = {}

    booster = xgb.train(
        params=best["params"],
        dtrain=dtrain,
        num_boost_round=int(best["best_round"]),
        evals=[(dtrain, "train"), (dtest, "test")],
        evals_result=evals_result,
        verbose_eval=False,
    )

    # Save eval history so your thesis can cite it
    with open(REPORTS_DIR / "training_evals_result.json", "w") as f:
        json.dump(evals_result, f, indent=2)

    # Make learning-curve plots (overfit/underfit evidence)
    _plot_learning_curves(evals_result, REPORTS_DIR)

    # ---- Evaluate on held-out test ----
    print("Evaluating on held-out test split...")
    y_prob = booster.predict(dtest)
    y_pred_05 = (y_prob >= 0.5).astype(int)
    y_pred_085 = (y_prob >= ENGAGEMENT_THRESHOLD).astype(int)

    metrics = {
        "auc_roc": float(roc_auc_score(y_test, y_prob)),
        "auc_pr": float(average_precision_score(y_test, y_prob)),

        "precision_0.5": float(precision_score(y_test, y_pred_05, zero_division=0)),
        "recall_0.5": float(recall_score(y_test, y_pred_05, zero_division=0)),
        "f1_0.5": float(f1_score(y_test, y_pred_05, zero_division=0)),

        "precision_0.85": float(precision_score(y_test, y_pred_085, zero_division=0)),
        "recall_0.85": float(recall_score(y_test, y_pred_085, zero_division=0)),
        "f1_0.85": float(f1_score(y_test, y_pred_085, zero_division=0)),

        "confusion_matrix_0.85": confusion_matrix(y_test, y_pred_085).tolist(),
        "threshold_used": ENGAGEMENT_THRESHOLD,

        "cv_best_auc": float(best["auc"]),
        "cv_best_round": int(best["best_round"]),
    }

    print("Test Metrics:")
    for k, v in metrics.items():
        print(f"  {k}: {v}")

    # ---- Save artifacts ----
    print("Saving artifacts...")
    joblib.dump(booster, ARTIFACTS_DIR / "model.joblib")

    with open(ARTIFACTS_DIR / "feature_columns.json", "w") as f:
        json.dump(feature_columns, f, indent=2)

    split_info = {
        "train_session_ids": train_df["session_id"].astype(str).tolist(),
        "test_session_ids": test_df["session_id"].astype(str).tolist(),
        "test_size": TEST_SIZE,
        "random_state": RANDOM_STATE,  
        "group_col": GROUP_COL,
        "label_col": LABEL_COL,
    }
    with open(ARTIFACTS_DIR / "split.json", "w") as f:
        json.dump(split_info, f, indent=2)

    metadata = {
        "trained_at": datetime.utcnow().isoformat(),
        "model": "xgboost.Booster",
        "xgboost_version": xgb.__version__,
        "n_features": len(feature_columns),
        "feature_columns": feature_columns,
        "label_col": LABEL_COL,
        "group_col": GROUP_COL,
        "test_size": TEST_SIZE,
        "random_state": RANDOM_STATE,
        "engagement_threshold": ENGAGEMENT_THRESHOLD,
        "drop_cols": DROP_COLS,
        "best_params": best["params"],
        "best_round": int(best["best_round"]),
        "metrics_test": metrics,
    }
    with open(ARTIFACTS_DIR / "metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)

    with open(REPORTS_DIR / "metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)

    print("Done.")
    print(f"Artifacts: {ARTIFACTS_DIR.resolve()}")
    print(f"Reports:   {REPORTS_DIR.resolve()}")


if __name__ == "__main__":
    main()

## `verification/engagement/xgboost/training/evaluate.py`
**What it does:** Standalone evaluation script for XGBoost on held-out test split.

In [ ]:
# SOURCE MIRROR FROM: verification/engagement/xgboost/training/evaluate.py
# PURPOSE: Standalone evaluation script for XGBoost on held-out test split.
# VIVA TIP: focus on function signatures, inputs/outputs, and why this file exists.

from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import xgboost as xgb

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay,
)
from sklearn.calibration import CalibrationDisplay


DATA_PATH = Path("data/processed/sessions_features.csv")
ARTIFACTS_DIR = Path("verification/engagement/xgboost/artifacts")
REPORTS_DIR = Path("reports")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

LABEL_COL = "engagement_label"
ENGAGEMENT_THRESHOLD = 0.85


def main():
    print("Loading artifacts...")
    model = joblib.load(ARTIFACTS_DIR / "model.joblib")

    with open(ARTIFACTS_DIR / "feature_columns.json") as f:
        feature_columns = json.load(f)

    with open(ARTIFACTS_DIR / "split.json") as f:
        split_info = json.load(f)

    test_session_ids = set(split_info["test_session_ids"])

    print("Loading dataset...")
    df = pd.read_csv(DATA_PATH)

    if "session_id" not in df.columns:
        raise ValueError("sessions_features.csv must contain session_id for split filtering.")

    df["session_id"] = df["session_id"].astype(str)
    test_df = df[df["session_id"].isin(test_session_ids)].copy()

    if test_df.empty:
        raise ValueError("Test split is empty after filtering. Check split.json and sessions_features.csv.")

    y = test_df[LABEL_COL].astype(int).values
    X = test_df[feature_columns].replace([np.inf, -np.inf], np.nan)
    X = X.fillna(X.median(numeric_only=True)).values

    print(f"Evaluating on test size: {len(test_df)}")

    dmat = xgb.DMatrix(X, feature_names=feature_columns)
    y_prob = model.predict(dmat)

    y_pred_05 = (y_prob >= 0.5).astype(int)
    y_pred_085 = (y_prob >= ENGAGEMENT_THRESHOLD).astype(int)

    metrics = {
        "auc_roc": float(roc_auc_score(y, y_prob)),
        "auc_pr": float(average_precision_score(y, y_prob)),

        "accuracy_0.5": float(accuracy_score(y, y_pred_05)),
        "precision_0.5": float(precision_score(y, y_pred_05, zero_division=0)),
        "recall_0.5": float(recall_score(y, y_pred_05, zero_division=0)),
        "f1_0.5": float(f1_score(y, y_pred_05, zero_division=0)),

        "accuracy_0.85": float(accuracy_score(y, y_pred_085)),
        "precision_0.85": float(precision_score(y, y_pred_085, zero_division=0)),
        "recall_0.85": float(recall_score(y, y_pred_085, zero_division=0)),
        "f1_0.85": float(f1_score(y, y_pred_085, zero_division=0)),

        "confusion_matrix_0.85": confusion_matrix(y, y_pred_085).tolist(),
        "threshold_used": ENGAGEMENT_THRESHOLD,
    }

    print("Metrics:")
    for k, v in metrics.items():
        print(f"{k}: {v}")

    with open(REPORTS_DIR / "metrics_test.json", "w") as f:
        json.dump(metrics, f, indent=2)

    # --- Confusion Matrix ---
    cm = confusion_matrix(y, y_pred_085)
    disp = ConfusionMatrixDisplay(cm)
    disp.plot()
    plt.title(f"Confusion Matrix (Threshold = {ENGAGEMENT_THRESHOLD}) – Test Split")
    plt.savefig(REPORTS_DIR / "confusion_matrix_085.png", bbox_inches="tight")
    plt.close()

    # --- ROC Curve ---
    RocCurveDisplay.from_predictions(y, y_prob)
    plt.title("ROC Curve – Test Split")
    plt.savefig(REPORTS_DIR / "roc_curve.png", bbox_inches="tight")
    plt.close()

    # --- PR Curve ---
    PrecisionRecallDisplay.from_predictions(y, y_prob)
    plt.title("Precision–Recall Curve – Test Split")
    plt.savefig(REPORTS_DIR / "pr_curve.png", bbox_inches="tight")
    plt.close()

    # --- Calibration plot ---
    CalibrationDisplay.from_predictions(y, y_prob, n_bins=10, strategy="quantile")
    plt.title("Calibration Plot – Test Split")
    plt.savefig(REPORTS_DIR / "calibration.png", bbox_inches="tight")
    plt.close()

    # --- Feature importance plot (gain) ---
    score = model.get_score(importance_type="gain")
    if score:
        imp = (
            pd.DataFrame({"feature": list(score.keys()), "gain": list(score.values())})
            .sort_values("gain", ascending=False)
            .head(20)
        )
        imp.plot(kind="barh", x="feature", y="gain", legend=False)
        plt.gca().invert_yaxis()
        plt.title("Top-20 Feature Importance (Gain)")
        plt.savefig(REPORTS_DIR / "feature_importance_gain.png", bbox_inches="tight")
        plt.close()

    # --- Accuracy / F1 vs Threshold (policy + gaming discussion) ---
    thresholds = np.linspace(0.05, 0.95, 19)
    rows = []
    for t in thresholds:
        yp = (y_prob >= t).astype(int)
        rows.append({
            "threshold": float(t),
            "accuracy": float(accuracy_score(y, yp)),
            "precision": float(precision_score(y, yp, zero_division=0)),
            "recall": float(recall_score(y, yp, zero_division=0)),
            "f1": float(f1_score(y, yp, zero_division=0)),
        })

    sweep = pd.DataFrame(rows)
    sweep.to_csv(REPORTS_DIR / "threshold_sweep.csv", index=False)

    plt.figure()
    plt.plot(sweep["threshold"], sweep["accuracy"], label="Accuracy")
    plt.plot(sweep["threshold"], sweep["f1"], label="F1")
    plt.xlabel("Threshold")
    plt.ylabel("Score")
    plt.title("Accuracy and F1 vs Threshold – Test Split")
    plt.legend()
    plt.savefig(REPORTS_DIR / "accuracy_f1_vs_threshold.png", bbox_inches="tight")
    plt.close()

    print(f"Reports saved to: {REPORTS_DIR.resolve()}")


if __name__ == "__main__":
    main()

## `verification/engagement/xgboost/inference/load.py`
**What it does:** Loads XGBoost model artifacts and required feature ordering.

In [ ]:
# SOURCE MIRROR FROM: verification/engagement/xgboost/inference/load.py
# PURPOSE: Loads XGBoost model artifacts and required feature ordering.
# VIVA TIP: focus on function signatures, inputs/outputs, and why this file exists.

import json
from pathlib import Path
import joblib
from app.core.settings import settings

ARTIFACTS_DIR = Path(settings.engagement_artifacts_xgboost_dir)

_model = None
_feature_columns = None
_metadata = None


def load_model():
    global _model
    if _model is None:
        model_path = ARTIFACTS_DIR / "model.joblib"
        if not model_path.exists():
            raise FileNotFoundError("model.joblib not found. Train the model first.")
        _model = joblib.load(model_path)
    return _model


def load_feature_columns():
    global _feature_columns
    if _feature_columns is None:
        path = ARTIFACTS_DIR / "feature_columns.json"
        if not path.exists():
            raise FileNotFoundError("feature_columns.json not found.")
        with open(path, "r") as f:
            _feature_columns = json.load(f)
    return _feature_columns


def load_metadata():
    global _metadata
    if _metadata is None:
        path = ARTIFACTS_DIR / "metadata.json"
        if not path.exists():
            _metadata = {}
        else:
            with open(path, "r") as f:
                _metadata = json.load(f)
    return _metadata

## `verification/engagement/xgboost/inference/predict.py`
**What it does:** Runs XGBoost inference for one session payload.

In [ ]:
# SOURCE MIRROR FROM: verification/engagement/xgboost/inference/predict.py
# PURPOSE: Runs XGBoost inference for one session payload.
# VIVA TIP: focus on function signatures, inputs/outputs, and why this file exists.

import numpy as np
import xgboost as xgb

from verification.engagement.xgboost.inference.load import load_model, load_feature_columns
from verification.engagement.common.validate import validate_features


def predict_engagement(features: dict):
    booster = load_model()  # xgboost.Booster
    feature_columns = load_feature_columns()

    validate_features(features, feature_columns)

    x = np.array([[features[col] for col in feature_columns]], dtype=float)
    dmat = xgb.DMatrix(x, feature_names=feature_columns)

    score = float(booster.predict(dmat)[0])

    return {
        "engagement_score": score,
    }

## `verification/engagement/xgboost/explain/shap_explain.py`
**What it does:** Computes local SHAP values and extracts top contributors.

In [ ]:
# SOURCE MIRROR FROM: verification/engagement/xgboost/explain/shap_explain.py
# PURPOSE: Computes local SHAP values and extracts top contributors.
# VIVA TIP: focus on function signatures, inputs/outputs, and why this file exists.

from __future__ import annotations

from typing import Dict, List, Tuple

import numpy as np
import shap
import xgboost as xgb

from verification.engagement.xgboost.inference.load import load_model, load_feature_columns

_explainer = None


def _get_explainer():
    global _explainer
    if _explainer is None:
        booster = load_model()  # xgboost.Booster
        _explainer = shap.TreeExplainer(booster)
    return _explainer


def compute_local_shap(features: Dict[str, float]) -> List[Dict[str, float]]:
    feature_columns = load_feature_columns()
    explainer = _get_explainer()

    # Default missing features to 0.0 rather than crash
    x = np.array([[float(features.get(col, 0.0)) for col in feature_columns]], dtype=float)
    dmat = xgb.DMatrix(x, feature_names=feature_columns)

    shap_vals = explainer.shap_values(dmat)
    shap_row = shap_vals[0]  # (n_features,)

    out: List[Dict[str, float]] = []
    for i, col in enumerate(feature_columns):
        out.append(
            {"feature": col, "value": float(x[0, i]), "shap": float(shap_row[i])}
        )
    return out


def top_contributors(
    shap_rows: List[Dict[str, float]],
    k: int = 3
) -> Tuple[List[Dict[str, float]], List[Dict[str, float]]]:
    if not shap_rows:
        return [], []
    sorted_rows = sorted(shap_rows, key=lambda r: r.get("shap", 0.0))
    top_negative = sorted_rows[:k]
    top_positive = list(reversed(sorted_rows[-k:]))
    return top_negative, top_positive

## `verification/engagement/ebm/training/train.py`
**What it does:** End-to-end EBM training pipeline with tuning, evaluation, and artifact export.

In [ ]:
# SOURCE MIRROR FROM: verification/engagement/ebm/training/train.py
# PURPOSE: End-to-end EBM training pipeline with tuning, evaluation, and artifact export.
# VIVA TIP: focus on function signatures, inputs/outputs, and why this file exists.

"""
train_ebm.py – Production-grade EBM training pipeline.

Mirrors the XGBoost pipeline in train.py but uses InterpretML's
ExplainableBoostingClassifier (glass-box GAM).

Training strategy:
  1. Same data prep / stratified-grouped split as XGBoost
  2. 40-trial random search (same approach as XGBoost) scored
     by mean CV AUC via StratifiedGroupKFold
  3. outer_bags=1 + max_rounds=500 for fast CV evaluation
  4. Final model retrained on full train split with best params
     (max_rounds=5000 + validation_size=0.15 for early stopping)
  5. Learning curves, calibration, and artifact dump
"""

from __future__ import annotations

import json
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)
from sklearn.model_selection import StratifiedGroupKFold

from interpret.glassbox import ExplainableBoostingClassifier

from verification.engagement.common.split import split_train_test

# ===========================
# CONFIG
# ===========================
DATA_PATH = Path("data/processed/sessions_features.csv")

LABEL_COL = "engagement_label"
GROUP_COL = "user_id"

ARTIFACTS_DIR = Path("verification/engagement/ebm/artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

REPORTS_DIR = Path("reports")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.2

ENGAGEMENT_THRESHOLD = 0.85

DROP_COLS = [
    LABEL_COL,
    GROUP_COL,
    "session_id",
    "video_id",
    "video_title",
]

# ===========================
# HYPERPARAMETER SEARCH SPACE
# ===========================
# Production approach: 40-trial random search (same as the XGBoost pipeline).
# EBM is more expensive per trial than XGBoost, so random search gives
# better coverage-per-minute than exhaustive grid.
# Final model uses max_rounds=5000 + early stopping; CV uses max_rounds=500.

PARAM_SPACE = {
    "max_bins":           [128, 256, 512],
    "learning_rate":      [0.005, 0.01, 0.02, 0.03, 0.05, 0.08],
    "max_leaves":         [2, 3, 4, 5, 6],
    "min_samples_leaf":   [2, 4, 5, 8, 10, 15],
    "interactions":       [0, 3, 5, 8, 10],
    "max_interaction_bins": [16, 32, 64],
}

N_RANDOM_TRIALS = 40  # matches XGBoost's 40-trial random search


def _prepare_features(df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray, List[str], pd.DataFrame]:
    """Identical to XGBoost pipeline — single source of truth would be ideal."""
    if LABEL_COL not in df.columns:
        raise ValueError(f"Missing label column: {LABEL_COL}")

    y = df[LABEL_COL].astype(int).values

    X = df.drop(columns=[c for c in DROP_COLS if c in df.columns], errors="ignore")
    X = X.select_dtypes(include=[np.number]).copy()

    # Median impute
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.fillna(X.median(numeric_only=True))

    feature_columns = list(X.columns)
    if not feature_columns:
        raise ValueError("No numeric features found after preprocessing.")

    return X.values, y, feature_columns, X


def _make_cv_folds(
    df_train: pd.DataFrame,
    feature_columns: List[str],
) -> List[Tuple[np.ndarray, np.ndarray]]:
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    y = df_train[LABEL_COL].astype(int).values
    groups = df_train[GROUP_COL].astype(str).values
    X_dummy = df_train[feature_columns].values

    folds = []
    for tr_idx, va_idx in sgkf.split(X_dummy, y, groups):
        folds.append((tr_idx, va_idx))
    return folds


def _sample_random_params(rng: np.random.Generator) -> Dict:
    """Sample one random hyperparameter configuration."""
    return {
        "max_bins": int(rng.choice(PARAM_SPACE["max_bins"])),
        "learning_rate": float(rng.choice(PARAM_SPACE["learning_rate"])),
        "max_leaves": int(rng.choice(PARAM_SPACE["max_leaves"])),
        "min_samples_leaf": int(rng.choice(PARAM_SPACE["min_samples_leaf"])),
        "interactions": int(rng.choice(PARAM_SPACE["interactions"])),
        "max_interaction_bins": int(rng.choice(PARAM_SPACE["max_interaction_bins"])),
    }


def _train_evaluate_fold(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
    feature_columns: List[str],
    params: Dict,
) -> float:
    """Train EBM on one fold, return validation AUC."""
    ebm = ExplainableBoostingClassifier(
        feature_names=feature_columns,
        max_bins=params["max_bins"],
        learning_rate=params["learning_rate"],
        max_leaves=params["max_leaves"],
        min_samples_leaf=params["min_samples_leaf"],
        interactions=params["interactions"],
        max_interaction_bins=params["max_interaction_bins"],
        max_rounds=500,            # capped for CV speed; final model uses 5000
        outer_bags=1,              # no bagging in CV (we handle splits ourselves)
        validation_size=0.0,       # we supply our own CV fold; no inner split
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    ebm.fit(X_train, y_train)

    y_prob = ebm.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, y_prob)
    return auc


def _plot_global_importances(ebm, feature_columns: List[str], out_dir: Path) -> None:
    """Plot top-20 EBM term importances (mean absolute score)."""
    importances = ebm.term_importances()
    names = ebm.term_names_

    # Build DataFrame for the main (non-interaction) terms
    imp_df = pd.DataFrame({"feature": names, "importance": importances})
    imp_df = imp_df.sort_values("importance", ascending=False).head(20)

    plt.figure(figsize=(10, 8))
    plt.barh(range(len(imp_df)), imp_df["importance"].values, align="center")
    plt.yticks(range(len(imp_df)), imp_df["feature"].values)
    plt.gca().invert_yaxis()
    plt.xlabel("Mean Absolute Score")
    plt.title("EBM – Top-20 Term Importances")
    plt.tight_layout()
    plt.savefig(out_dir / "ebm_term_importances.png", bbox_inches="tight", dpi=150)
    plt.close()


def main():
    print("=" * 60)
    print("EBM Training Pipeline (Production-Grade)")
    print("=" * 60)

    # ---- 1. Load & prep ----
    print("\n[1/6] Loading dataset...")
    df = pd.read_csv(DATA_PATH)

    if LABEL_COL not in df.columns:
        raise ValueError(f"Label column '{LABEL_COL}' not found in dataset")
    if GROUP_COL not in df.columns:
        raise ValueError(f"Group column '{GROUP_COL}' not found in dataset")

    X_all, y_all, feature_columns, X_df = _prepare_features(df)

    clean_df = X_df.copy()
    clean_df[LABEL_COL] = y_all
    clean_df[GROUP_COL] = df[GROUP_COL].astype(str)
    clean_df["session_id"] = (
        df["session_id"].astype(str) if "session_id" in df.columns else df.index.astype(str)
    )

    print(f"  Features: {len(feature_columns)}")
    print(f"  Samples:  {len(df)}")
    print(f"  Label distribution: {pd.Series(y_all).value_counts().to_dict()}")

    # ---- 2. Split ----
    print("\n[2/6] Splitting train/test (stratified + grouped)...")
    train_df, test_df = split_train_test(
        clean_df,
        label_col=LABEL_COL,
        group_col=GROUP_COL,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
    )

    X_train = train_df[feature_columns].values
    y_train = train_df[LABEL_COL].values
    X_test = test_df[feature_columns].values
    y_test = test_df[LABEL_COL].values

    print(f"  Train: {len(train_df)} | Test: {len(test_df)}")

    # ---- 3. Hyperparameter random search with CV ----
    print(f"\n[3/6] Random search ({N_RANDOM_TRIALS} trials × 5-fold CV)...")
    folds = _make_cv_folds(train_df, feature_columns)
    rng = np.random.default_rng(RANDOM_STATE)
    n_total = N_RANDOM_TRIALS

    best = {"mean_auc": -1.0, "params": None, "all_results": []}

    for i in range(1, n_total + 1):
        params = _sample_random_params(rng)
        fold_aucs = []

        for fold_idx, (tr_idx, va_idx) in enumerate(folds):
            X_tr_fold = X_train[tr_idx]
            y_tr_fold = y_train[tr_idx]
            X_va_fold = X_train[va_idx]
            y_va_fold = y_train[va_idx]

            try:
                auc = _train_evaluate_fold(
                    X_tr_fold, y_tr_fold, X_va_fold, y_va_fold,
                    feature_columns, params,
                )
                fold_aucs.append(auc)
            except Exception as e:
                print(f"    [!] Fold {fold_idx} failed for combo {i}: {e}")
                fold_aucs.append(0.0)

        mean_auc = float(np.mean(fold_aucs))
        std_auc = float(np.std(fold_aucs))

        result_entry = {
            "trial": i,
            "params": params,
            "mean_auc": mean_auc,
            "std_auc": std_auc,
            "fold_aucs": fold_aucs,
        }
        best["all_results"].append(result_entry)

        if mean_auc > best["mean_auc"]:
            best["mean_auc"] = mean_auc
            best["params"] = params
            print(f"  Trial {i}/{n_total}: ★ NEW BEST  AUC={mean_auc:.4f} ± {std_auc:.4f}  {params}")
        elif i % 50 == 0:
            print(f"  Trial {i}/{n_total}: AUC={mean_auc:.4f} (best so far: {best['mean_auc']:.4f})")

    if best["params"] is None:
        raise RuntimeError("Grid search failed: no valid param combination found.")

    print(f"\n  ✓ Best CV AUC: {best['mean_auc']:.4f}")
    print(f"  ✓ Best params: {best['params']}")

    # ---- 4. Train final model on full train split ----
    print("\n[4/6] Training final EBM on full train split...")
    final_ebm = ExplainableBoostingClassifier(
        feature_names=feature_columns,
        max_bins=best["params"]["max_bins"],
        learning_rate=best["params"]["learning_rate"],
        max_leaves=best["params"]["max_leaves"],
        min_samples_leaf=best["params"]["min_samples_leaf"],
        interactions=best["params"]["interactions"],
        max_interaction_bins=best["params"]["max_interaction_bins"],
        max_rounds=5000,
        early_stopping_rounds=50,
        validation_size=0.15,      # inner early-stop split for final training
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    final_ebm.fit(X_train, y_train)

    # ---- 5. Evaluate on held-out test ----
    print("\n[5/6] Evaluating on held-out test split...")
    y_prob = final_ebm.predict_proba(X_test)[:, 1]
    y_pred_05 = (y_prob >= 0.5).astype(int)
    y_pred_085 = (y_prob >= ENGAGEMENT_THRESHOLD).astype(int)

    metrics = {
        "model": "EBM (ExplainableBoostingClassifier)",
        "auc_roc": float(roc_auc_score(y_test, y_prob)),
        "auc_pr": float(average_precision_score(y_test, y_prob)),

        "precision_0.5": float(precision_score(y_test, y_pred_05, zero_division=0)),
        "recall_0.5": float(recall_score(y_test, y_pred_05, zero_division=0)),
        "f1_0.5": float(f1_score(y_test, y_pred_05, zero_division=0)),

        "precision_0.85": float(precision_score(y_test, y_pred_085, zero_division=0)),
        "recall_0.85": float(recall_score(y_test, y_pred_085, zero_division=0)),
        "f1_0.85": float(f1_score(y_test, y_pred_085, zero_division=0)),

        "confusion_matrix_0.85": confusion_matrix(y_test, y_pred_085).tolist(),
        "threshold_used": ENGAGEMENT_THRESHOLD,

        "cv_best_auc": best["mean_auc"],
    }

    print("\n  Test Metrics:")
    for k, v in metrics.items():
        print(f"    {k}: {v}")

    # ---- 6. Save artifacts ----
    print("\n[6/6] Saving artifacts...")

    # Model
    joblib.dump(final_ebm, ARTIFACTS_DIR / "ebm_model.joblib")

    # Feature columns
    with open(ARTIFACTS_DIR / "ebm_feature_columns.json", "w") as f:
        json.dump(feature_columns, f, indent=2)

    # Split info (shared format)
    split_info = {
        "train_session_ids": train_df["session_id"].astype(str).tolist(),
        "test_session_ids": test_df["session_id"].astype(str).tolist(),
        "test_size": TEST_SIZE,
        "random_state": RANDOM_STATE,
        "group_col": GROUP_COL,
        "label_col": LABEL_COL,
    }
    with open(ARTIFACTS_DIR / "ebm_split.json", "w") as f:
        json.dump(split_info, f, indent=2)

    # Metadata (full audit trail)
    # Serialize all_results: convert numpy types and trim fold_aucs for size
    serializable_results = []
    for r in best["all_results"]:
        serializable_results.append({
            "trial": r["trial"],
            "params": {k: (int(v) if isinstance(v, (np.integer,)) else float(v) if isinstance(v, (np.floating,)) else v) for k, v in r["params"].items()},
            "mean_auc": round(r["mean_auc"], 6),
            "std_auc": round(r["std_auc"], 6),
        })

    # Keep only top-10 trials in metadata to avoid huge files
    top_10_results = sorted(serializable_results, key=lambda x: x["mean_auc"], reverse=True)[:10]

    metadata = {
        "trained_at": datetime.utcnow().isoformat(),
        "model": "ExplainableBoostingClassifier",
        "interpret_version": ">=0.6.4",
        "n_features": len(feature_columns),
        "feature_columns": feature_columns,
        "label_col": LABEL_COL,
        "group_col": GROUP_COL,
        "test_size": TEST_SIZE,
        "random_state": RANDOM_STATE,
        "engagement_threshold": ENGAGEMENT_THRESHOLD,
        "drop_cols": DROP_COLS,
        "best_params": best["params"],
        "cv_best_auc": round(best["mean_auc"], 6),
        "total_trials": N_RANDOM_TRIALS,
        "search_strategy": "random",
        "top_10_trials": top_10_results,
        "metrics_test": metrics,
        "n_terms": len(final_ebm.term_names_),
        "term_names": list(final_ebm.term_names_),
    }
    with open(ARTIFACTS_DIR / "ebm_metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)

    # Reports
    with open(REPORTS_DIR / "ebm_metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)

    # Global importance plot
    _plot_global_importances(final_ebm, feature_columns, REPORTS_DIR)

    print("\n" + "=" * 60)
    print("✓ EBM training complete.")
    print(f"  Artifacts: {ARTIFACTS_DIR.resolve()}")
    print(f"  Reports:   {REPORTS_DIR.resolve()}")
    print("=" * 60)


if __name__ == "__main__":
    main()

## `verification/engagement/ebm/training/evaluate.py`
**What it does:** Standalone evaluation script for EBM on held-out test split.

In [ ]:
# SOURCE MIRROR FROM: verification/engagement/ebm/training/evaluate.py
# PURPOSE: Standalone evaluation script for EBM on held-out test split.
# VIVA TIP: focus on function signatures, inputs/outputs, and why this file exists.

"""
evaluate_ebm.py – Full evaluation suite for the EBM model.

Mirrors evaluate.py but with EBM-specific additions:
  - Global term importances
  - Per-feature shape function plots
  - Standard: ROC, PR, calibration, confusion matrix, threshold sweep
"""

from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay,
)
from sklearn.calibration import CalibrationDisplay


DATA_PATH = Path("data/processed/sessions_features.csv")
ARTIFACTS_DIR = Path("verification/engagement/ebm/artifacts")
REPORTS_DIR = Path("reports")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

LABEL_COL = "engagement_label"
ENGAGEMENT_THRESHOLD = 0.85


def _plot_shape_functions(ebm, out_dir: Path, max_plots: int = 12) -> None:
    """
    Plot top-N EBM shape functions (partial-dependence-like curves).
    These are the glass-box internals: y = Σ f_i(x_i).
    Only 1D main-effect terms are plotted; interaction terms are skipped.
    """
    importances = ebm.term_importances()

    # Collect only plottable 1D main-effect terms
    plottable = []
    for term_idx in range(len(importances)):
        scores = ebm.term_scores_[term_idx]
        if scores.ndim != 1:
            continue  # skip 2D interaction terms
        bins_list = ebm.bins_[term_idx]
        if not bins_list or len(bins_list) == 0:
            continue
        plottable.append((term_idx, importances[term_idx]))

    if not plottable:
        print("  [!] No plottable 1D shape functions found.")
        return

    # Sort by importance descending, take top N
    plottable.sort(key=lambda x: x[1], reverse=True)
    plottable = plottable[:max_plots]
    n_plots = len(plottable)

    fig, axes = plt.subplots(
        nrows=(n_plots + 2) // 3,
        ncols=3,
        figsize=(18, 4 * ((n_plots + 2) // 3)),
    )
    axes = axes.flatten() if n_plots > 1 else [axes]

    for idx, (term_idx, imp) in enumerate(plottable):
        ax = axes[idx]
        term_name = ebm.term_names_[term_idx]
        bins = ebm.bins_[term_idx][0]
        scores = ebm.term_scores_[term_idx]

        # bins: cut-point edges; scores may include extra entries for
        # missing/unknown. Build x from bins and trim scores to match.
        x_vals = np.concatenate([[bins[0] - 1], bins])
        y_vals = scores[: len(x_vals)]

        ax.step(x_vals, y_vals, where="post", linewidth=1.5)
        ax.set_xlabel(term_name)
        ax.set_ylabel("Score contribution")
        ax.set_title(f"{term_name}\n(imp={imp:.3f})")
        ax.axhline(0, color="gray", linestyle="--", alpha=0.5)

    # Hide unused axes
    for j in range(len(plottable), len(axes)):
        axes[j].set_visible(False)

    plt.suptitle("EBM Shape Functions (Top Features)", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.savefig(out_dir / "ebm_shape_functions.png", bbox_inches="tight", dpi=150)
    plt.close()


def main():
    print("=" * 60)
    print("EBM Evaluation Pipeline")
    print("=" * 60)

    # ---- Load artifacts ----
    print("\nLoading EBM artifacts...")
    ebm = joblib.load(ARTIFACTS_DIR / "ebm_model.joblib")

    with open(ARTIFACTS_DIR / "ebm_feature_columns.json") as f:
        feature_columns = json.load(f)

    with open(ARTIFACTS_DIR / "ebm_split.json") as f:
        split_info = json.load(f)

    test_session_ids = set(split_info["test_session_ids"])

    # ---- Load dataset ----
    print("Loading dataset...")
    df = pd.read_csv(DATA_PATH)

    if "session_id" not in df.columns:
        raise ValueError("sessions_features.csv must contain session_id for split filtering.")

    df["session_id"] = df["session_id"].astype(str)
    test_df = df[df["session_id"].isin(test_session_ids)].copy()

    if test_df.empty:
        raise ValueError("Test split is empty. Check ebm_split.json and sessions_features.csv.")

    y = test_df[LABEL_COL].astype(int).values
    X = test_df[feature_columns].replace([np.inf, -np.inf], np.nan)
    X = X.fillna(X.median(numeric_only=True)).values

    print(f"Evaluating on test size: {len(test_df)}")

    # ---- Predict ----
    y_prob = ebm.predict_proba(X)[:, 1]
    y_pred_05 = (y_prob >= 0.5).astype(int)
    y_pred_085 = (y_prob >= ENGAGEMENT_THRESHOLD).astype(int)

    metrics = {
        "model": "EBM",
        "auc_roc": float(roc_auc_score(y, y_prob)),
        "auc_pr": float(average_precision_score(y, y_prob)),

        "accuracy_0.5": float(accuracy_score(y, y_pred_05)),
        "precision_0.5": float(precision_score(y, y_pred_05, zero_division=0)),
        "recall_0.5": float(recall_score(y, y_pred_05, zero_division=0)),
        "f1_0.5": float(f1_score(y, y_pred_05, zero_division=0)),

        "accuracy_0.85": float(accuracy_score(y, y_pred_085)),
        "precision_0.85": float(precision_score(y, y_pred_085, zero_division=0)),
        "recall_0.85": float(recall_score(y, y_pred_085, zero_division=0)),
        "f1_0.85": float(f1_score(y, y_pred_085, zero_division=0)),

        "confusion_matrix_0.85": confusion_matrix(y, y_pred_085).tolist(),
        "threshold_used": ENGAGEMENT_THRESHOLD,
    }

    print("\nMetrics:")
    for k, v in metrics.items():
        print(f"  {k}: {v}")

    with open(REPORTS_DIR / "ebm_metrics_test.json", "w") as f:
        json.dump(metrics, f, indent=2)

    # --- Confusion Matrix ---
    cm = confusion_matrix(y, y_pred_085)
    disp = ConfusionMatrixDisplay(cm)
    disp.plot()
    plt.title(f"EBM – Confusion Matrix (Threshold = {ENGAGEMENT_THRESHOLD})")
    plt.savefig(REPORTS_DIR / "ebm_confusion_matrix_085.png", bbox_inches="tight")
    plt.close()

    # --- ROC Curve ---
    RocCurveDisplay.from_predictions(y, y_prob)
    plt.title("EBM – ROC Curve")
    plt.savefig(REPORTS_DIR / "ebm_roc_curve.png", bbox_inches="tight")
    plt.close()

    # --- PR Curve ---
    PrecisionRecallDisplay.from_predictions(y, y_prob)
    plt.title("EBM – Precision–Recall Curve")
    plt.savefig(REPORTS_DIR / "ebm_pr_curve.png", bbox_inches="tight")
    plt.close()

    # --- Calibration plot ---
    CalibrationDisplay.from_predictions(y, y_prob, n_bins=10, strategy="quantile")
    plt.title("EBM – Calibration Plot")
    plt.savefig(REPORTS_DIR / "ebm_calibration.png", bbox_inches="tight")
    plt.close()

    # --- Global importance ---
    importances = ebm.term_importances()
    names = ebm.term_names_
    imp_df = (
        pd.DataFrame({"feature": names, "importance": importances})
        .sort_values("importance", ascending=False)
        .head(20)
    )
    imp_df.plot(kind="barh", x="feature", y="importance", legend=False)
    plt.gca().invert_yaxis()
    plt.title("EBM – Top-20 Term Importances")
    plt.savefig(REPORTS_DIR / "ebm_feature_importance.png", bbox_inches="tight")
    plt.close()

    # --- Shape function plots ---
    try:
        _plot_shape_functions(ebm, REPORTS_DIR, max_plots=12)
    except Exception as e:
        print(f"  [!] Shape function plots skipped: {e}")

    # --- Threshold sweep ---
    thresholds = np.linspace(0.05, 0.95, 19)
    rows = []
    for t in thresholds:
        yp = (y_prob >= t).astype(int)
        rows.append({
            "threshold": float(t),
            "accuracy": float(accuracy_score(y, yp)),
            "precision": float(precision_score(y, yp, zero_division=0)),
            "recall": float(recall_score(y, yp, zero_division=0)),
            "f1": float(f1_score(y, yp, zero_division=0)),
        })

    sweep = pd.DataFrame(rows)
    sweep.to_csv(REPORTS_DIR / "ebm_threshold_sweep.csv", index=False)

    plt.figure()
    plt.plot(sweep["threshold"], sweep["accuracy"], label="Accuracy")
    plt.plot(sweep["threshold"], sweep["f1"], label="F1")
    plt.xlabel("Threshold")
    plt.ylabel("Score")
    plt.title("EBM – Accuracy and F1 vs Threshold")
    plt.legend()
    plt.savefig(REPORTS_DIR / "ebm_accuracy_f1_vs_threshold.png", bbox_inches="tight")
    plt.close()

    print(f"\n✓ EBM evaluation complete. Reports: {REPORTS_DIR.resolve()}")


if __name__ == "__main__":
    main()

## `verification/engagement/ebm/inference/load.py`
**What it does:** Loads EBM model artifacts and required feature ordering.

In [ ]:
# SOURCE MIRROR FROM: verification/engagement/ebm/inference/load.py
# PURPOSE: Loads EBM model artifacts and required feature ordering.
# VIVA TIP: focus on function signatures, inputs/outputs, and why this file exists.

import json
from pathlib import Path
import joblib
from app.core.settings import settings

ARTIFACTS_DIR = Path(settings.engagement_artifacts_ebm_dir)

_ebm_model = None
_ebm_feature_columns = None


def load_ebm_model():
    global _ebm_model
    if _ebm_model is None:
        model_path = ARTIFACTS_DIR / "ebm_model.joblib"
        if not model_path.exists():
            raise FileNotFoundError("ebm_model.joblib not found. Run train_ebm first.")
        _ebm_model = joblib.load(model_path)
    return _ebm_model


def load_ebm_feature_columns():
    global _ebm_feature_columns
    if _ebm_feature_columns is None:
        path = ARTIFACTS_DIR / "ebm_feature_columns.json"
        if not path.exists():
            raise FileNotFoundError("ebm_feature_columns.json not found.")
        with open(path, "r") as f:
            _ebm_feature_columns = json.load(f)
    return _ebm_feature_columns

## `verification/engagement/ebm/inference/predict.py`
**What it does:** Runs EBM inference for one session payload.

In [ ]:
# SOURCE MIRROR FROM: verification/engagement/ebm/inference/predict.py
# PURPOSE: Runs EBM inference for one session payload.
# VIVA TIP: focus on function signatures, inputs/outputs, and why this file exists.

"""
predict_ebm.py – EBM inference (scikit-learn interface, no DMatrix).
"""
from __future__ import annotations

import numpy as np

from verification.engagement.ebm.inference.load import load_ebm_model, load_ebm_feature_columns
from verification.engagement.common.validate import validate_features


def predict_engagement_ebm(features: dict) -> dict:
    """
    Predict engagement using the trained EBM model.

    Returns the same shape as predict_engagement() for XGBoost so the
    API layer can treat both interchangeably:
      {"engagement_score": float}
    """
    ebm = load_ebm_model()
    feature_columns = load_ebm_feature_columns()

    validate_features(features, feature_columns)

    x = np.array([[features[col] for col in feature_columns]], dtype=float)

    score = float(ebm.predict_proba(x)[0, 1])

    return {
        "engagement_score": score,
    }

## `verification/engagement/ebm/explain/ebm_explain.py`
**What it does:** Computes local EBM contributions and extracts top contributors.

In [ ]:
# SOURCE MIRROR FROM: verification/engagement/ebm/explain/ebm_explain.py
# PURPOSE: Computes local EBM contributions and extracts top contributors.
# VIVA TIP: focus on function signatures, inputs/outputs, and why this file exists.

"""
ebm_explain.py – Production-grade native EBM explanations (glass-box, exact).

EBM is a Generalized Additive Model: prediction = Σ f_i(x_i) + intercept.
Each f_i(x_i) is the *exact* contribution of feature i for the given input.
No SHAP approximation needed – the explanation IS the model.

Key differences from SHAP-based explanations:
  • Contributions are exact (not a Shapley value approximation).
  • We can verify: intercept + Σ contributions ≡ logit prediction (sanity check).
  • Interaction terms are explicit and split equally to their component features.

Interface matches shap_explain.py for seamless API-level swapping.
"""

from __future__ import annotations

import logging
from typing import Dict, List, Tuple

import numpy as np

from verification.engagement.ebm.inference.load import load_ebm_model, load_ebm_feature_columns

log = logging.getLogger(__name__)


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def _extract_score(raw_score) -> float:
    """
    EBM explain_local() returns term scores that can be either:
      - a plain float (scalar)
      - a per-class array, e.g. [neg_class_score, pos_class_score]

    For binary classification we always want the positive-class (index 1)
    contribution, which is in log-odds space.
    """
    if hasattr(raw_score, "__len__") and len(raw_score) > 1:
        return float(raw_score[1])
    return float(raw_score)


def _extract_intercept(raw_intercept) -> float:
    """Same per-class handling for the intercept term."""
    if hasattr(raw_intercept, "__len__") and len(raw_intercept) > 1:
        return float(raw_intercept[1])
    return float(raw_intercept)


# ---------------------------------------------------------------------------
# Core: compute local EBM contributions
# ---------------------------------------------------------------------------

def compute_local_ebm(features: Dict[str, float]) -> List[Dict[str, float]]:
    """
    Compute exact local contributions from the EBM model.

    Returns a list matching shap_explain.compute_local_shap() output:
      [{"feature": str, "value": float, "shap": float}, ...]

    We re-use the "shap" key name so the downstream pipeline
    (text_explainer, API schemas) works unchanged.
    The value is the EBM term score – the exact contribution in log-odds
    space (not a SHAP approximation).
    """
    ebm = load_ebm_model()
    feature_columns = load_ebm_feature_columns()

    x = np.array(
        [[float(features.get(col, 0.0)) for col in feature_columns]],
        dtype=float,
    )

    # --- Extract local explanation -----------------------------------------
    local_explanation = ebm.explain_local(x)
    data = local_explanation.data(0)

    term_names = data["names"]
    raw_scores = data["scores"]
    raw_intercept = data.get("intercept", 0.0)

    # Robust extraction – handles both scalar and per-class arrays
    term_scores = [_extract_score(s) for s in raw_scores]
    intercept = _extract_intercept(raw_intercept)

    # --- Glass-box sanity check --------------------------------------------
    # The entire point of using EBM: we can PROVE the explanation is exact.
    # intercept + Σ term_scores  MUST equal the model's logit prediction.
    computed_logit = intercept + sum(term_scores)

    try:
        predicted_logit = float(ebm.decision_function(x)[0])
        drift = abs(computed_logit - predicted_logit)

        if drift < 1e-4:
            log.debug(
                "[EBM-SANITY-OK] intercept=%.4f  Σscores=%.4f  "
                "computed_logit=%.4f  predicted_logit=%.4f  drift=%.6f",
                intercept, sum(term_scores),
                computed_logit, predicted_logit, drift,
            )
        else:
            log.warning(
                "[EBM-SANITY-FAIL] Explanation drift %.6f exceeds tolerance!  "
                "intercept=%.4f  Σscores=%.4f  computed=%.4f  predicted=%.4f",
                drift, intercept, sum(term_scores),
                computed_logit, predicted_logit,
            )
    except AttributeError:
        # Some EBM versions may not expose decision_function;
        # skip check gracefully.
        log.debug(
            "[EBM-SANITY-SKIP] decision_function not available; "
            "sanity check skipped.  intercept=%.4f  Σscores=%.4f",
            intercept, sum(term_scores),
        )

    # --- Map term scores to feature columns --------------------------------
    # EBM terms include main effects and optionally interaction terms
    # like "feature_a x feature_b".  We split interactions 50/50 (standard
    # heuristic) so the output aligns with feature_columns.
    contribution_map: Dict[str, float] = {}
    has_interactions = False

    for name, score in zip(term_names, term_scores):
        name_str = str(name)
        if " x " in name_str:
            # Interaction term — split contribution equally
            has_interactions = True
            parts = [p.strip() for p in name_str.split(" x ")]
            share = score / len(parts)
            for part in parts:
                contribution_map[part] = contribution_map.get(part, 0.0) + share
        else:
            contribution_map[name_str] = (
                contribution_map.get(name_str, 0.0) + score
            )

    if has_interactions:
        log.debug(
            "[EBM-INTERACTIONS] Interaction terms detected and split "
            "equally to component features."
        )

    # --- Build output (same shape as shap_explain) -------------------------
    out: List[Dict[str, float]] = []
    for col in feature_columns:
        out.append({
            "feature": col,
            "value": float(features.get(col, 0.0)),
            "shap": float(contribution_map.get(col, 0.0)),
        })

    return out


# ---------------------------------------------------------------------------
# Top contributors (same interface as shap_explain.top_contributors)
# ---------------------------------------------------------------------------

def top_contributors_ebm(
    ebm_rows: List[Dict[str, float]],
    k: int = 3,
) -> Tuple[List[Dict[str, float]], List[Dict[str, float]]]:
    """
    Same interface as shap_explain.top_contributors().
    Returns (top_negative, top_positive) sorted by contribution magnitude.
    """
    if not ebm_rows:
        return [], []
    sorted_rows = sorted(ebm_rows, key=lambda r: r.get("shap", 0.0))
    top_negative = sorted_rows[:k]
    top_positive = list(reversed(sorted_rows[-k:]))
    return top_negative, top_positive